In [2]:
!pip install autogenstudio

# Install AutoGen Studio and pyngrok
!pip install -q autogenstudio pyngrok

# Verify the installation
!autogenstudio version

# ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken

import os
import getpass

# 1. Set up Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Set up Ngrok Auth Token (Required to tunnel the AutoGen Studio UI)
NGROK_TOKEN =""
os.environ["NGROK_TOKEN"] = NGROK_TOKEN
!ngrok config add-authtoken {NGROK_TOKEN}

gsk_rvS22WBFWqqirkXaakzoWGdyb3FYKXnIp0mkviVfsbZavGKxKI8SRequirement already satisfied: autogenstudio in /usr/local/lib/python3.12/dist-packages (0.4.2.2)
AutoGen Studio  CLI version: 0.4.2.2
Enter your Groq API Key: ··········
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:

#For running Autogen Studio in google colab

import multiprocessing
import time
import os
from pyngrok import ngrok, conf


def run_autogen_studio():
    # Launch AutoGen Studio on port 8081
    !autogenstudio ui --port 8081 --host 0.0.0.0


# Start AutoGen Studio in a background process
process = multiprocessing.Process(target=run_autogen_studio)
process.start()

# Wait a moment for the server to spin up
time.sleep(5)

# Set the ngrok authtoken explicitly for pyngrok
NGROK_TOKEN = os.environ.get("NGROK_TOKEN")
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print("NGROK_TOKEN environment variable not found. Please ensure it's set in a previous cell.")

# Open the ngrok tunnel to port 8081
public_url = ngrok.connect(8081)

print("\n" + "="*60)
print(f"[SUCCESS] AutoGen Studio is running!")
print(f"Click the link below to open the UI:")
print(f"{public_url}")
print("="*60 + "\n")



[SUCCESS] AutoGen Studio is running!
Click the link below to open the UI:
NgrokTunnel: "https://endorse-grunge-unlovely.ngrok-free.dev" -> "http://localhost:8081"



In [4]:
import os

import asyncio

from autogen_agentchat.agents import AssistantAgent

from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_agentchat.conditions import MaxMessageTermination

from autogen_ext.models.openai import OpenAIChatCompletionClient



async def run_team():

    # Fetch the Groq API key you securely entered in Cell 2

    groq_key = ""



    # Define Groq model clients with the correct model_info settings

    researcher_model = OpenAIChatCompletionClient(

        model="llama-3.1-8b-instant",

        base_url="https://api.groq.com/openai/v1",

        api_key=groq_key,

        model_info={

            "vision": False,

            "function_calling": True,

            "json_output": True,

            "structured_output": True,

            "family": "unknown"

        }

    )



    editor_model = OpenAIChatCompletionClient(

        model="llama-3.3-70b-versatile",

        base_url="https://api.groq.com/openai/v1",

        api_key=groq_key,

        model_info={

            "vision": False,

            "function_calling": True,

            "json_output": True,

            "structured_output": True,

            "family": "unknown"

        }

    )



    # Define the individual Agents

    researcher = AssistantAgent(

        name="Researcher",

        model_client=researcher_model,

        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."

    )



    editor = AssistantAgent(

        name="Editor",

        model_client=editor_model,

        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."

    )



    # Orchestrate the workflow team

    team = RoundRobinGroupChat(

        participants=[researcher, editor],

        termination_condition=MaxMessageTermination(max_messages=4)

    )



    # Run the prompt

    print("--- Starting Multi-Agent Session ---")

    async for message in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):

        print(f"\n\033[1m[{message.sender}]\033[0m: {message.content}")

        print("-" * 40)



# Execute the async loop inside Google Colab

await run_team()

--- Starting Multi-Agent Session ---


AttributeError: 'TextMessage' object has no attribute 'sender'

In [5]:
import os

from autogen_agentchat.agents import AssistantAgent

from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_agentchat.ui import Console

from autogen_ext.models.openai import OpenAIChatCompletionClient

from autogen_agentchat.conditions import  TextMentionTermination



groq_key = os.environ.get("GROQ_API_KEY", "") # Retrieve the key if set in the environment



agent = AssistantAgent(

        name="weather_agent",

        model_client=OpenAIChatCompletionClient(

            model="gpt-4o-mini",

            api_key=groq_key,

            base_url="https://api.groq.com/openai/v1" # Assuming Groq is the desired backend

        ),

    )

agent_team = RoundRobinGroupChat([agent], termination_condition=TextMentionTermination("TERMINATE"))

config = agent_team.dump_component()

print(config.model_dump_json())

{"provider":"autogen_agentchat.teams.RoundRobinGroupChat","component_type":"team","version":1,"component_version":1,"description":"A team that runs a group chat with participants taking turns in a round-robin fashion\n    to publish a message to all.","label":"RoundRobinGroupChat","config":{"participants":[{"provider":"autogen_agentchat.agents.AssistantAgent","component_type":"agent","version":1,"component_version":1,"description":"An agent that provides assistance with tool use.","label":"AssistantAgent","config":{"name":"weather_agent","model_client":{"provider":"autogen_ext.models.openai.OpenAIChatCompletionClient","component_type":"model","version":1,"component_version":1,"description":"Chat completion client for OpenAI hosted models.","label":"OpenAIChatCompletionClient","config":{"model":"gpt-4o-mini","api_key":"**********","base_url":"https://api.groq.com/openai/v1"}},"workbench":{"provider":"autogen_core.tools.StaticWorkbench","component_type":"workbench","version":1,"component

In [6]:
import math

def calculator(expression: str) -> str:
    """Evaluate a safe math expression and return the result."""
    try:
        allowed = {k: getattr(math, k) for k in dir(math)}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

In [7]:
# Skill: get_stock_price (stub — swap for a real API)
import requests

def get_stock_price(ticker: str) -> str:
    """Return the latest price for a stock ticker."""
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    r = requests.get(url, timeout=5)
    data = r.json()
    price = data["chart"]["result"][0]["meta"]["regularMarketPrice"]
    return f"{ticker}: ${price}

SyntaxError: unterminated f-string literal (detected at line 10) (1815558887.py, line 10)